# Congklak AlphaZero Training (Stable Version)

### ⚠️ IMPORTANT: Set Runtime to GPU
1. Click **Runtime** -> **Change runtime type**.
2. Select **T4 GPU**.
3. Click **Save**.

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Set up the Environment (Git + Drive Hybrid)
import os, sys, torch

# 1. Define paths
repo_url = "https://github.com/billdansr/AlphaDDA.git"  # Your fork!
local_repo_path = "/content/AlphaDDA"
game_subdir = "AlphaZero/Congklak"
drive_path = "/content/drive/MyDrive/Colab Notebooks/AlphaZero/Congklak"

# 2. Sync Code from GitHub
print("--- SYNCING CODE ---")
if not os.path.exists(local_repo_path):
    print(f"Cloning fork from {repo_url}...")
    !git clone {repo_url} {local_repo_path}
else:
    print("Updating existing repository...")
    %cd {local_repo_path}
    !git pull
    %cd /content

# 3. Sync Models from Google Drive
print("\n--- SYNCING MODELS ---")
local_game_path = os.path.join(local_repo_path, game_subdir)
os.chdir(local_game_path)

# Copy latest models from Drive to local workspace for faster training
!cp -v "{drive_path}"/*.model . 2>/dev/null || print("No existing models found in Drive. Starting fresh.")

# 4. Verify Hardware
print(f"\nCUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# Step 3: Start Training
# We also copy the models back to Drive automatically after it finishes
%env PYTHONPATH=.:$PYTHONPATH

!python train_mp.py

print("Training finished or interrupted. Saving models back to Drive...")
import glob
import shutil

for f in glob.glob('*.model'):
    print(f'Copying {f} to Drive...')
    shutil.copy(f, drive_path)

if not glob.glob('*.model'):
    print('No model files found. Training may have failed or not reached the first checkpoint.')

### Manual Backup
Run this if you want to save models manually at any time.

In [ ]:
!cp -v *.model "/content/drive/MyDrive/Colab Notebooks/AlphaZero/Congklak/"